# Model training and edge export

This notebook documents the model-selection work: recording-level train/test splitting, feature ablation, challenge-set evaluation, and conversion of scaler/model values into firmware constants.

> **Reproducible workflow:** Use `python -m training.train_model` for the maintained training pipeline. This notebook remains available for step-by-step inspection of the experiments.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score
)
import pandas as pd

In [ ]:
features_df = pd.read_csv("../../data/raw/final/processed/features.csv")

In [ ]:
feature_columns = [
    "rms",
    "ptp",
    "dominant_freq",
    "energy_low",
    "energy_mid",
    "energy_high"
]

train_df = features_df[
    ~features_df["recording_id"].str.endswith("_3")
].copy()

test_df = features_df[
    features_df["recording_id"].str.endswith("_3")
].copy()

print("Train:")
print(train_df.groupby("condition").size())

print("\nTest:")
print(test_df.groupby("condition").size())

In [ ]:
X_train = train_df[feature_columns]
y_train = train_df["label"]

X_test = test_df[feature_columns]
y_test = test_df["label"]

In [ ]:
model = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "classifier",
        LogisticRegression(
            max_iter=1000
        )
    )
])


In [ ]:
model.fit(X_train, y_train)  

In [ ]:
predictions = model.predict(
    X_test
)

In [ ]:
print(
    "Accuracy:",
    accuracy_score(
        y_test,
        predictions
    )
)

In [ ]:
print(
    "\nConfusion Matrix:"
)

print(
    confusion_matrix(
        y_test,
        predictions
    )
)

In [ ]:
print(
    "\nClassification Report:"
)

print(
    classification_report(
        y_test,
        predictions
    )
)

In [ ]:
def train_and_evaluate(
    train_df,
    test_df,
    feature_columns
):

    X_train = train_df[feature_columns]
    y_train = train_df["label"]

    X_test = test_df[feature_columns]
    y_test = test_df["label"]

    model = Pipeline([
        (
            "scaler",
            StandardScaler()
        ),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000
            )
        )
    ])

    model.fit(
        X_train,
        y_train
    )

    predictions = model.predict(
        X_test
    )

    accuracy = accuracy_score(
        y_test,
        predictions
    )

    return model, accuracy

In [ ]:
time_features = [
    "rms",
    "ptp"
]

frequency_features = [
    "dominant_freq",
    "energy_low",
    "energy_mid",
    "energy_high"
]

all_features = [
    "rms",
    "ptp",
    "dominant_freq",
    "energy_low",
    "energy_mid",
    "energy_high"
]

In [ ]:
time_model, time_accuracy = train_and_evaluate(
    train_df,
    test_df,
    time_features
)

freq_model, freq_accuracy = train_and_evaluate(
    train_df,
    test_df,
    frequency_features
)

full_model, full_accuracy = train_and_evaluate(
    train_df,
    test_df,
    all_features
)

print(
    "Time-domain only:",
    time_accuracy
)

print(
    "Frequency-domain only:",
    freq_accuracy
)

print(
    "All features:",
    full_accuracy
)

In [ ]:
rms_model, rms_accuracy = train_and_evaluate(
    train_df,
    test_df,
    ["rms"]
)

print("RMS only:", rms_accuracy)

In [ ]:
features_df.groupby("condition")[
    [
        "rms",
        "ptp",
        "dominant_freq"
    ]
].agg(["min", "max", "mean"])

In [ ]:
# For stationary data, we can use the same features and model. Let's load the stationary data and evaluate the model.

challenge_df = pd.read_csv("../../data/raw/final/processed/challenge.csv")

In [ ]:
challenge_df["prediction"] = full_model.predict(
    challenge_df[feature_columns]
)

In [ ]:
challenge_df["anomaly_probability"] = (
    full_model.predict_proba(
        challenge_df[feature_columns]
    )[:, 1]
)


In [ ]:
print(
    challenge_df[
        [
            "recording_id",
            "condition",
            "window",
            "rms",
            "dominant_freq",
            "prediction",
            "anomaly_probability"
        ]
    ]
)

In [ ]:
print(
    challenge_df.groupby("condition")[
        [
            "rms",
            "dominant_freq",
            "anomaly_probability"
        ]
    ].agg(
        ["mean", "min", "max"]
    )
)

In [ ]:
print(
    challenge_df
    .groupby(
        ["condition", "prediction"]
    )
    .size()
)

In [ ]:
# DAY 4

In [ ]:
experiments = {
    "RMS only": [
        "rms"
    ],

    "Time domain": [
        "rms",
        "ptp"
    ],

    "Frequency domain": [
        "dominant_freq",
        "energy_low",
        "energy_mid",
        "energy_high"
    ],

    "All features": [
        "rms",
        "ptp",
        "dominant_freq",
        "energy_low",
        "energy_mid",
        "energy_high"
    ]
}

results = {}

for name, cols in experiments.items():

    model, accuracy = train_and_evaluate(
        train_df,
        test_df,
        cols
    )

    results[name] = accuracy

    print(
        name,
        "->",
        accuracy
    )

In [ ]:
normal_train = train_df[
    train_df["condition"] == "normal_40"
]

print(
    normal_train["rms"].describe()
)

In [ ]:
print(
    "Normal minimum:",
    normal_train["rms"].min()
)

print(
    "Normal maximum:",
    normal_train["rms"].max()
)

In [ ]:
scaler = full_model.named_steps["scaler"]

classifier = full_model.named_steps[
    "classifier"
]

In [ ]:
print("Scaler means:")

for feature, mean in zip(
    feature_columns,
    scaler.mean_
):
    print(
        feature,
        "->",
        mean
    )

In [ ]:
print(
    "\nScaler scales:"
)

for feature, scale in zip(
    feature_columns,
    scaler.scale_
):
    print(feature, scale)

In [ ]:
print(
    "\nCoefficients:"
)

for feature, coefficient in zip(
    feature_columns,
    classifier.coef_[0]
):
    print(
        feature,
        coefficient
    )

print(
    "\nIntercept:",
    classifier.intercept_[0]
)

In [ ]:
normal_min_rms = normal_train["rms"].min()

idle_threshold = (
    0.5 * normal_min_rms
)

print(
    "Candidate idle threshold:",
    idle_threshold
)

In [ ]:
import numpy as np
def manual_logistic_prediction(features):

    means = np.array([
        0.2503907708505175,
        0.3384182347313635,
        47.75,
        59.42973099986822,
        343.5047411290585,
        1109.5851043311964
    ])

    scales = np.array([
        0.11362929225264064,
        0.17608072223841076,
        9.743587634952538,
        87.5033657620107,
        375.466718476449,
        1290.270961050111
    ])

    weights = np.array([
        1.5033973064737176,
        1.353159283810666,
        0.38334747727525725,
        0.3139286200415267,
        1.7203284350690617,
        0.4626510277901541
    ])

    intercept = 2.419389700061559

    standardized = (
        features - means
    ) / scales

    score = (
        np.dot(
            weights,
            standardized
        )
        + intercept
    )

    probability = (
        1 / (1 + np.exp(-score))
    )

    return probability

In [ ]:
row = test_df.iloc[0]

x = row[feature_columns].to_numpy(
    dtype=float
)

manual_probability = (
    manual_logistic_prediction(x)
)

sklearn_probability = (
    full_model.predict_proba(
        row[feature_columns]
        .to_frame()
        .T
    )[0, 1]
)

print(
    "Manual:",
    manual_probability
)

print(
    "Scikit-learn:",
    sklearn_probability
)

In [ ]:
experiments = {
    "RMS only": [
        "rms"
    ],

    "Time domain": [
        "rms",
        "ptp"
    ],

    "Frequency domain": [
        "dominant_freq",
        "energy_low",
        "energy_mid",
        "energy_high"
    ],

    "All features": [
        "rms",
        "ptp",
        "dominant_freq",
        "energy_low",
        "energy_mid",
        "energy_high"
    ]
}

for name, cols in experiments.items():

    model, accuracy = train_and_evaluate(
        train_df,
        test_df,
        cols
    )

    print(name, ":", accuracy)

In [ ]:
edge_features = [
    "rms",
    "ptp",
]

In [ ]:
edge_model, edge_accuracy = train_and_evaluate(
    train_df,
    test_df,
    edge_features
)

print(
    "Edge model accuracy:",
    edge_accuracy
)

In [ ]:
edge_scaler = edge_model.named_steps[
    "scaler"
]

edge_classifier = edge_model.named_steps[
    "classifier"
]


print("MEANS")

for feature, value in zip(
    edge_features,
    edge_scaler.mean_
):
    print(feature, value)


print("\nSCALES")

for feature, value in zip(
    edge_features,
    edge_scaler.scale_
):
    print(feature, value)


print("\nWEIGHTS")

for feature, value in zip(
    edge_features,
    edge_classifier.coef_[0]
):
    print(feature, value)


print(
    "\nINTERCEPT:",
    edge_classifier.intercept_[0]
)

In [ ]:
challenge_df["edge_prediction"] = (
    edge_model.predict(
        challenge_df[edge_features]
    )
)

challenge_df["edge_anomaly_probability"] = (
    edge_model.predict_proba(
        challenge_df[edge_features]
    )[:, 1]
)

In [ ]:
print(
    challenge_df[
        [
            "recording_id",
            "condition",
            "window",
            "rms",
            "ptp",
            "edge_prediction",
            "edge_anomaly_probability"
        ]
    ]
)

In [ ]:
print(
    challenge_df
    .groupby(
        ["condition", "edge_prediction"]
    )
    .size()
)

In [ ]:
edge_scaler = edge_model.named_steps["scaler"]
edge_classifier = edge_model.named_steps["classifier"]

print("MEANS")
for feature, value in zip(
    ["rms", "ptp"],
    edge_scaler.mean_
):
    print(feature, value)

print("\nSCALES")
for feature, value in zip(
    ["rms", "ptp"],
    edge_scaler.scale_
):
    print(feature, value)

print("\nWEIGHTS")
for feature, value in zip(
    ["rms", "ptp"],
    edge_classifier.coef_[0]
):
    print(feature, value)

print(
    "\nINTERCEPT:",
    edge_classifier.intercept_[0]
)

In [ ]:
"""MEANS
rms 0.2503907708505175
ptp 0.3384182347313635

SCALES
rms 0.11362929225264064
ptp 0.17608072223841076

WEIGHTS
rms 2.3543119819921503
ptp 2.107691551908885

INTERCEPT: 3.4872158011716263"""